# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [2]:
#A way to check if the imports are working

import os  # Should be instant
print("os imported")

import json  # Should be instant
print("json imported")

from dotenv import load_dotenv  # Should be instant
print("dotenv imported")

from IPython.display import Markdown, display, update_display  # Should be instant
print("IPython imported")

from openai import OpenAI  # Might take 1-2 seconds max
print("openai imported")

from scraper import fetch_website_links, fetch_website_contents  # This is the suspect
print("scraper imported")

os imported
json imported
dotenv imported
IPython imported
openai imported
scraper imported


In [11]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import json
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [12]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [4]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/1

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [13]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [14]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/

In [15]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [9]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'linkedin page', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [16]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [17]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano


KeyboardInterrupt: 

In [12]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 3 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'company page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [18]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [14]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 9 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Qwen/Qwen3-Coder-Next
Updated
7 days ago
•
141k
•
689
zai-org/GLM-OCR
Updated
1 day ago
•
373k
•
907
openbmb/MiniCPM-o-4_5
Updated
about 4 hours ago
•
30.4k
•
709
moonshotai/Kimi-K2.5
Updated
5 days ago
•
504k
•
1.96k
ACE-Step/Ace-Step1.5
Updated
7 days ago
•
28.7k
•
490
Browse 2M+ models
Spaces
Running
on
A100
Featured
240
ACE-Step v1.5
🎵
240
Music Generation Foundation Model v1.5
Running
on
Zero
Featured
1.35k
Qwen3-TTS Demo
🎙
1.35k
Generate speech from text with voice design, cloning, or speakers
Running
535
Demo Playground
⚡
535
F

In [19]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
 You are an assistant that analyzes the contents of several relevant pages from a company website
 and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
 Respond in markdown without code blocks.
 Include details of company culture, customers and careers/jobs if you have the information.
 """

In [20]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [21]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 8 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nzai-org/GLM-5\nUpdated\n4 days ago\n•\n168k\n•\n1.28k\nMiniMaxAI/MiniMax-M2.5\nUpdated\n1 day ago\n•\n31.6k\n•\n688\nQwen/Qwen3.5-397B-A17B\nUpdated\n1 day ago\n•\n19.6k\n•\n558\nNanbeige/Nanbeige4.1-3B\nUpdated\nabout 6 hours ago\n•\n32k\n•\n524\nopenbmb/MiniCPM-SALA\nUpdated\n6 days ago\n•\n3.86k\n•\n457\nBrowse 2M+ models\nSpaces\nRunning\n756\nDemo Playground\n⚡\n756\nFree platform to a

In [18]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [19]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


# Welcome to Hugging Face: The Friendliest AI Community on the Planet! 🤗

---

## Who Are We?

Hugging Face is *the* AI community-building hub where machine learning enthusiasts—from fresh-faced newbies to brilliant scientists—assemble to create the future. Think of us as the *town square* of AI, where you can explore, collaborate, and share over **2 million models**, **500k+ datasets**, and *countless* applications across text, image, video, audio, and even 3D.

---

## What’s the Buzz About?

- **Models galore:** From photorealistic image creators to speech streaming wizards, our ever-trending models come fresh out of the AI oven daily.
- **Datasets:** Need data? We have over half a million datasets waiting to fuel your next ML masterpiece.
- **Spaces:** Showcase your AI apps or try out featured ones like the “Wan2.2 Animate” or “Qwen Image 2512,” all running on *zero hassle*.
- **Community:** Join a vibrant, growing crowd of machine learners shaping ethical and open AI.

---

## Why Hugging Face? (Besides the Hug Emoji)

- **Collaboration Station:** Host unlimited public models, datasets, and applications — because sharing is caring.
- **Build Your Portfolio:** Put your ML skills on display and gain street cred in the AI world.
- **Move Faster:** Our open-source stack lets you accelerate development — no turbo button required.
- **Enterprise & Compute:** Serious AI work? We’ve got paid solutions to keep your projects powered and performant.

---

## Our Culture: Warm, Welcoming & Wonderfully Witty

- We *love* open-source and openness (except for secrets to our killer AI jokes).
- We believe in *learning through sharing* — everyone is both a student and a teacher here.
- Our community is as diverse as the models we host — from hobbyists to researchers, we’re all friends here.
- Collaboration beats competition, and coffee beats everything.

---

## For Curious Customers, Investors & Future Hugging Face Fans

- **Customers:** Say goodbye to reinventing the wheel. Find pre-trained models and datasets to jumpstart your AI projects with ease and speed.
- **Investors:** Join a thriving ecosystem pushing the AI frontier ethically and collaboratively.
- **Job Seekers:** Dream of pushing AI to new heights with impact? Hugging Face is where your passion meets cutting-edge tech and a community that cheers you on.

---

## Join the Hugging Face Family!

Explore AI apps, browse millions of models, datasets, and applications — or better yet, create your own and share it with the world! Sign up today and be a part of the AI revolution that’s as friendly as it is futuristic.

---

**Hugging Face** — where AI is not just smart, it's *hug*smart! 🤗

---

### Brand Colors to Brighten Your Day  
- Sunny Yellow #FFD21E  
- Warm Orange #FF9D00  
- Smooth Gray #6B7280

Want to see our smiling logo? Check it out on the site — it’s almost as friendly as our community.  

---

**Psst!** Ready to accelerate your machine learning journey? The future’s bright, the models are hot, and the datasets are plentiful. Come build it with Hugging Face!

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [19]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [20]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 5 relevant links


# Welcome to Hugging Face: Where AI Meets Community (and We Hug It Out!)

---

## Who We Are — The AI Party You Don’t Want to Miss

Hugging Face isn’t just a name. We are **the AI community building the future** — a buzzing hive where machine learning enthusiasts, experts, and innovators from all around the globe gather to play, build, and shake hands (virtually, but with the same enthusiasm). 

Our mission? Make machine learning *more human* by creating a platform where sharing genius models, juicy datasets, and groundbreaking applications happens every day – like AI’s version of a potluck, but with 2 million+ models and counting!

---

## What’s Cooking? Models, Datasets & Spaces Galore!

- **Models:** Your AI toolkit – over **2 million models** ready to flex their neural muscles. From the latest coding wizards like **Qwen3-Coder-Next** to image generation magicians like **Z Image Turbo**, our library is the AI candy store of your dreams.
  
- **Datasets:** Nothing feeds your AI hungry brains better. Dive into half a million datasets spanning math, text, images, video, and even 3D.
  
- **Spaces:** Your playground for running and showcasing AI apps. Fancy generating music, animations, or text-to-speech demos? Spaces is your stage powered by NVIDIA’s mighty A100 GPUs or zero-cost "Zero" compute options.

---

## Why Join the Hug? Company Culture Highlights 

- **Collaboration is our love language:** We thrive on open-source sharing, fostering connections and accelerating innovation worldwide.
  
- **All modalities, all the time:** Be it text, images, or audio, diversity sparks creativity here.
  
- **Careers with a cause:** Whether you want to build models, manage data, engineer powerful platforms, or support enterprise clients — your skills will flourish in a culture emphasizing growth and impact.

- **Fun meets functionality:** We work hard, and play well by crafting a supportive community where your AI experiments and wildest ideas are encouraged.

---

## For the Enterprises: Power, Privacy & Peace of Mind

Big teams, big dreams? Hugging Face’s **Enterprise Hub** offers a fortress of features:

- Enterprise-grade **security** with Single Sign-On, audit logs, and granular access control — because your models deserve the best bodyguards.
  
- **Advanced compute options** like ZeroGPU quota boosts keep your teams zooming past bottlenecks.
  
- Flexible billing, priority support, **private dataset viewers**, and more to keep your AI engine humming smoothly.
  
- Pricing that starts at a reasonable **$20/user/month** for teams, with custom contracts for enterprise giants.

---

## Our Customers: From Solo Researchers to AI Titans

Hugging Face is the favorite playground for:

- Data scientists crafting next-gen models
- Enterprises scaling AI responsibly and securely
- Educators, hobbyists, and AI artists seeking to share and shine
- Developers embedding the magic of AI apps into real-world products

If you work with AI, chances are YOU belong here.

---

## Careers: Join the Revolution

Looking for a job where your code can literally *change the future?* Hugging Face offers:

- Roles across engineering, data science, product, research, and developer advocacy
- A chance to build and maintain infrastructure that hosts millions of models and datasets
- An inclusive, open, curious culture with plenty of room to grow and hug your peers (virtually… for now)

Ready to rocket AI forward? [Join Us!](https://huggingface.co/careers)

---

## Parting Hug

Whether you're an AI newbie, a hardcore researcher, a startup builder, or a corporate fleet captain — Hugging Face is your AI home base. Join us where innovation is a team sport, models are endless, and the future isn't just built, it’s warmly HUGGED together.

**Sign up, dive in, and let’s build the future — one model, dataset, and smile at a time!**

---

*Hugging Face: Machine Learning’s Friendliest Neighborhood*  
[Explore Hugging Face →](https://huggingface.co) | [Enterprise Solutions →](https://huggingface.co/enterprise) | [Careers →](https://huggingface.co/careers)  

---

*P.S. No actual hugging required, but a good sense of humor and passion for AI highly recommended!*

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

###Make brochure with gradio

In [38]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr # oh yeah!
from scraper import fetch_website_contents

In [29]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AIzaSyCP


In [30]:
openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [39]:
system_message = """
 You are an assistant that analyzes the contents of several relevant pages from a company website
 and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
 Respond in markdown without code blocks.
 Include details of company culture, customers and careers/jobs if you have the information.
 """

In [40]:
def stream_gpt(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = openai.chat.completions.create(
        model='gpt-4.1-mini',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [41]:
def stream_claude(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = anthropic.chat.completions.create(
        model='claude-sonnet-4-5-20250929',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [42]:
def stream_brochure(company_name, url, model):
    yield ""
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt += fetch_website_contents(url)
    if model=="GPT":
        result = stream_gpt(prompt)
    elif model=="Claude":
        result = stream_claude(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [43]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for the LLM", lines=7)
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
model_selector = gr.Dropdown(["GPT", "Claude"], label="Select model", value="GPT")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure,
    title="LLMs", 
    inputs=[message_input, url_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["Give some websites to scrape", "https://edwarddonner.com", "GPT"],
            ["Explain the Transformer architecture to an aspiring AI engineer", "Claude"]
        ], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7877
* To create a public link, set `share=True` in `launch()`.


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>